In [48]:
import tifffile as tiff
from pathlib import Path
import pandas as pd

import torch
import torchvision.transforms.functional as TF

Loading the data for preprocessing

In [49]:
DataPath = Path("..") / "Training data"
SavePath = Path("..") / "Processed data"

train_tabular = pd.read_csv(DataPath / "train_tabular.csv")
print(f"Tabular shape: {train_tabular.shape}")

Tabular shape: (1024, 23)


Deal with missing data

In [50]:
#Finding missing data
missing_data = train_tabular.isnull().sum()
missing_data = missing_data[missing_data > 0]
print("Columns with missing data:")
print(missing_data)

sentinel_files = set()
viirs_files = set()

for idx, row in train_tabular.iterrows():
    sentinel_files.add(row['sentinel2_tiff_file_name'])
    viirs_files.add(row['viirs_tiff_file_name'])

print(f"Total referenced sentinel files: {len(sentinel_files)}")
print(f"Total referenced viirs files: {len(viirs_files)}")

train_composite_path = DataPath / "train_composite"

for file in train_composite_path.iterdir():
    if file.name in sentinel_files:
        sentinel_files.remove(file.name)
    elif file.name in viirs_files:
        viirs_files.remove(file.name)
    else:
        print(f"Unreferenced file: {file.name}")

print(f"Missing sentinel files: {len(sentinel_files)}")
print(f"Missing viirs files: {len(viirs_files)}")

print("Data integrity check complete.")

Columns with missing data:
tropical_cyclone_wind_risk    4
dtype: int64
Total referenced sentinel files: 1024
Total referenced viirs files: 1024
Missing sentinel files: 0
Missing viirs files: 0
Data integrity check complete.


In [51]:
#Print rows with missing data
if not missing_data.empty:
    print("Rows with missing data:")
    #Row will consist of geolocation_name, quarter_label, year, and tropical_cyclone_wind_risk
    rows_with_missing = train_tabular[train_tabular.isnull().any(axis=1)][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
    print(rows_with_missing)

    #Make a set of unique geolocation names with missing data
    unique_geolocations_with_missing = set(rows_with_missing['geolocation_name'].unique())

    #Other rows with the same geolocation_name
    for geolocation in unique_geolocations_with_missing:
        similar_rows = train_tabular[train_tabular['geolocation_name'] == geolocation][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
        similar_rows.sort_values(by=['year', 'quarter_label'], inplace=True)
        print(f"Other rows with geolocation_name {geolocation}:")
        print(similar_rows)
        #Get the most common categorical value tropical_cyclone_wind_risk value for these rows
        most_common_value = similar_rows['tropical_cyclone_wind_risk'].mode()[0]
        print(f"Most common tropical_cyclone_wind_risk value for {geolocation}: {most_common_value}")
        #Fill missing tropical_cyclone_wind_risk values with the most common value
        train_tabular.loc[(train_tabular['geolocation_name'] == geolocation) & (train_tabular['tropical_cyclone_wind_risk'].isnull()), 'tropical_cyclone_wind_risk'] = most_common_value

Rows with missing data:
    geolocation_name quarter_label  year tropical_cyclone_wind_risk
239      25000 Shiga       2022-Q4  2022                        NaN
382      25000 Shiga       2022-Q1  2022                        NaN
460      25000 Shiga       2022-Q3  2022                        NaN
743      25000 Shiga       2022-Q2  2022                        NaN
Other rows with geolocation_name 25000 Shiga:
    geolocation_name quarter_label  year tropical_cyclone_wind_risk
101      25000 Shiga       2019-Q1  2019                   Moderate
915      25000 Shiga       2019-Q3  2019                   Moderate
188      25000 Shiga       2019-Q4  2019                   Moderate
81       25000 Shiga       2020-Q2  2020                        Low
226      25000 Shiga       2020-Q3  2020                        Low
520      25000 Shiga       2020-Q4  2020                        Low
106      25000 Shiga       2021-Q1  2021                   Moderate
547      25000 Shiga       2021-Q2  2021      

Convert categorical columns to numeric

In [52]:
numeric_columns = train_tabular.select_dtypes(include=['number']).columns
print(f"Numeric columns: {numeric_columns.tolist()}")
nonnumeric_columns = train_tabular.select_dtypes(exclude=['number']).columns
print(f"Nonnumeric columns before processing: {nonnumeric_columns.tolist()}")

#Convert geolocation_id into numeric format
location_mapping = {loc: idx for idx, loc in enumerate(train_tabular['geolocation_name'].unique())}
train_tabular['geolocation_name'] = train_tabular['geolocation_name'].map(location_mapping).astype('Int64')

#Convert quarter_labels into a numeric format like 2020.25 for 2020-Q1, 2019.5 for 2019-Q2, etc.
quarter_mapping = {}
for row in train_tabular['quarter_label'].unique():
    quarter_mapping[row] = int(row.split('-')[1][1])

train_tabular['quarter_label'] = train_tabular['quarter_label'].map(quarter_mapping)
#print(f"Unique quarter labels: {train_tabular['quarter_label'].nunique()}")
#print("Columns after processing:")
#print(train_tabular['quarter_label'].head())

#Convert yes/no columns to 1/0
yes_no_columns = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'flood_risk_class']
for col in yes_no_columns:
    train_tabular[col] = train_tabular[col].map({'Yes': 1, 'No': 0}).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())

#Convert country to "japan" = 0 and "philipines" = 1
train_tabular["country"] = train_tabular["country"].map({'Philippines': 0, 'Japan': 1}).astype('Int64')


#Convert region_economic_classification to numeric codes
economic_map = {
    'Low income': 0,
    'Lower-middle income': 1,
    'Upper-middle income': 2,
    'High income': 3
}
train_tabular['region_economic_classification'] = train_tabular['region_economic_classification'].map(economic_map).astype('Int64')
#print(f"Converted 'region_economic_classification' to numeric codes.")
#print(train_tabular['region_economic_classification'].head())

risk_columns = ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']
risk_map = {
    'Very Low': 0,
    'Low': 1,
    'Moderate': 2,
    'High': 3,
    'Very High': 4
}
for col in risk_columns:
    train_tabular[col] = train_tabular[col].map(risk_map).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())

train_tabular['koppen_climate_zone'] = train_tabular['koppen_climate_zone'].astype('category').cat.codes.astype('Int64')

nonnumeric_columns = train_tabular.select_dtypes(exclude=['number']).columns
categorical_columns = train_tabular.columns.difference(numeric_columns).difference(nonnumeric_columns)
print(f"Nonnumeric columns: {nonnumeric_columns.tolist()}")
print(f"Categorical columns: {categorical_columns.tolist()}")

print(train_tabular.columns)

Numeric columns: ['year', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km', 'construction_cost_per_m2_usd']
Nonnumeric columns before processing: ['data_id', 'geolocation_name', 'quarter_label', 'country', 'developed_country', 'landlocked', 'region_economic_classification', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'seismic_hazard_zone', 'flood_risk_class', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk', 'koppen_climate_zone', 'sentinel2_tiff_file_name', 'viirs_tiff_file_name']
Nonnumeric columns: ['data_id', 'sentinel2_tiff_file_name', 'viirs_tiff_file_name']
Categorical columns: ['access_to_airport', 'access_to_highway', 'access_to_port', 'access_to_railway', 'country', 'developed_country', 'flood_risk_class', 'geolocation_name', 'koppen_climate_zone', 'landlocked', 'quarter_label', 'region_economic_classification', 'seismic_hazard_zone', 'tornadoes_wind_risk', 'tropical_cyclone_wind_risk']
Index(['data_id', 'geolocation_name

Image processing

In [53]:
def load_tiff(path):
        img = tiff.imread(path)
        t = torch.tensor(img, dtype=torch.float32)
        if t.dim() == 2:
            t = t.unsqueeze(0)
        elif t.dim() == 3:
            t = t.permute(2, 0, 1)
        t = torch.nan_to_num(t, nan=0.0)
        t = TF.resize(t, [224, 224])
        return t

def normalize_image(t):
    mean = t.mean(dim=(1, 2), keepdim=True)
    std = t.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
    return (t - mean) / std

def process_image(sentinel_path, viirs_path):
    sentinel_img = load_tiff(sentinel_path)
    viirs_img = load_tiff(viirs_path)

    sentinel_img = normalize_image(sentinel_img)
    viirs_img = normalize_image(viirs_img)

    return {'sentinel': sentinel_img, 'viirs': viirs_img}

counter = 0
images = train_tabular[['sentinel2_tiff_file_name', 'viirs_tiff_file_name']].itertuples(index=False)
len_images = len(train_tabular)
processed_files = []

for sentinel, viirs in images:
    counter += 1
    print(f"Processing image {counter} of {len_images}")

    sentinelFileName = "_".join(sentinel.split('.')[0].split('_')[2:])
    viirsFileName = "_".join(viirs.split('.')[0].split('_')[1:])
    
    if sentinelFileName == viirsFileName:
        newFileName = sentinelFileName + ".pt"
        print(f"Processing {newFileName}...")
        tensor = process_image(DataPath / "train_composite" / sentinel, DataPath / "train_composite" / viirs)
        torch.save(tensor, SavePath / "processed_composite" / newFileName)
        processed_files.append(newFileName)
    else:
        print(f"Warning: Sentinel file {sentinel} and VIIRS file {viirs} do not match after processing. Skipping.")
        processed_files.append(None)

train_tabular['processed_imgs'] = processed_files

Processing image 1 of 1024
Processing dinagat_islands_2019-Q3.pt...
Processing image 2 of 1024
Processing 29000_nara_2024-Q2.pt...
Processing image 3 of 1024
Processing 05000_akita_2020-Q1.pt...
Processing image 4 of 1024
Processing cotabato_2020-Q4.pt...
Processing image 5 of 1024
Processing pampanga_2019-Q3.pt...
Processing image 6 of 1024
Processing antique_2019-Q1.pt...
Processing image 7 of 1024
Processing 17000_ishikawa_2021-Q2.pt...
Processing image 8 of 1024
Processing 22000_shizuoka_2021-Q3.pt...
Processing image 9 of 1024
Processing 21000_gifu_2020-Q2.pt...
Processing image 10 of 1024
Processing catanduanes_2019-Q1.pt...
Processing image 11 of 1024
Processing 43000_kumamoto_2024-Q2.pt...
Processing image 12 of 1024
Processing 45000_miyazaki_2019-Q3.pt...
Processing image 13 of 1024
Processing 46000_kagoshima_2021-Q2.pt...
Processing image 14 of 1024
Processing leyte_2019-Q1.pt...
Processing image 15 of 1024
Processing 03000_iwate_2024-Q3.pt...
Processing image 16 of 1024
Proc

Since the data is split into two countries Japan and Philippines, the data is split to train two separate models. For each country to enhance model performance.

In [54]:
# Split the data based on country
philipines = pd.DataFrame()
japan = pd.DataFrame()

for row in train_tabular.itertuples(index=False):
    if row.country == "Japan" or row.country == 1:
        japan = pd.concat([japan, pd.DataFrame(data=[row])], ignore_index=True)
    elif row.country == "Philippines" or row.country == 0:
        philipines = pd.concat([philipines, pd.DataFrame(data=[row])], ignore_index=True)
    else:
        raise ValueError(f"Unknown country: {row.country}")

print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

All data shape: (1024, 24)
Philippines shape: (457, 24)
Japan shape: (567, 24)


Normalizing data

In [55]:
def normalize(df : pd.DataFrame, normalizing_cols  : list) -> pd.DataFrame:

    for col in normalizing_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val - min_val > 0:
            df[col] = (df[col] - min_val) / (max_val - min_val)
        else:
            df[col] = 0.0
    return df

"""Normalizing the data"""
normalizing_cols = ['year', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km']
philipines = normalize(philipines, normalizing_cols)
japan = normalize(japan, normalizing_cols)
train_tabular = normalize(train_tabular, normalizing_cols)


Saving the data after preprocessing to use for model training.

In [56]:
#Drop colums with only one unique value
def drop_constant_columns(df):
    for col in df.columns:
        if df[col].nunique() == 1:
            df = df.drop(columns=[col])
    return df

def drop_redundant_columns(df):
    redundant = ['sentinel2_tiff_file_name', 'viirs_tiff_file_name']
    for col in redundant:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df

def remove_matching_columns(df):
    exiting_cols = []
    for col in df:
        if col in exiting_cols:
            continue
        for other_col in df:
            if col == other_col:
                continue
            if (df[col] == df[other_col]).all():
                print(f"Column: {col}, matches column: {other_col}")
                exiting_cols.append(other_col)
    df = df.drop(columns=exiting_cols)
    return df

def process(df):
    df = drop_constant_columns(df)
    df = drop_redundant_columns(df)
    df = remove_matching_columns(df)
    return df

train_tabular = process(train_tabular)
philipines = process(philipines)
japan = process(japan)

print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

train_tabular.to_csv(SavePath / "processed_data.csv", index=False)
philipines.to_csv(SavePath / "processed_philippines.csv", index=False)
japan.to_csv(SavePath / "processed_japan.csv", index=False)

Column: country, matches column: developed_country
All data shape: (1024, 21)
Philippines shape: (457, 20)
Japan shape: (567, 18)
